# Human RefSeq transcript coding-status ML study

**Question:** Can simple sequence features distinguish human RefSeq transcripts labeled by accession prefix as protein-coding (`NM_`) versus non-coding (`NR_`)? FLIP2 and variant information are intentionally out of scope.

This notebook downloads the specified NCBI Datasets package, audits `NM_`/`NR_` labels against GFF3, reports every `NR_` GFF3 biotype, groups isoforms by `GeneID`, clusters one representative per gene with MMseqs2 at **80% nucleotide identity and 80% coverage**, and performs a cluster-level train/validation/test split. It retains all `NR_` biotypes, including short RNAs, and reports results by biotype.

## 0. Assumptions and minimum-length rule

The operational label is `NM_ = 1` and `NR_ = 0`; it is a RefSeq convention, not an independent functional assay. The only length rule is **`length >= 20 nt`**, used as a malformed/empty-sequence QC floor, not a biological filter. It is deliberately low enough to retain miRNA, tRNA, snRNA, and other legitimate short RNAs; length is reported because it can otherwise become a shortcut.

In [ ]:
import os, re, sys, json, gzip, shutil, zipfile, subprocess, random
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np, pandas as pd
SEED=42; ACCESSION="GCF_000001405.40"; EXPECTED_ASSEMBLY="GRCh38.p14"
EXPECTED_ANNOTATION_RELEASE="GCF_000001405.40-RS_2025_08"
MIN_LENGTH_NT=20; MAX_MODEL_PER_CLASS=3000; TEST_SIZE=.20; VAL_SIZE=.20
MMSEQS_IDENTITY=.80; MMSEQS_COVERAGE=.80
DATA_DIR=Path("ncbi_refseq_cache"); DATA_DIR.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print(datetime.now(timezone.utc).isoformat(), sys.version.split()[0])
def ensure(cmd, apt):
    if shutil.which(cmd) is None:
        subprocess.run(["apt-get","update","-qq"],check=True); subprocess.run(["apt-get","install","-y","-qq",apt],check=True)
ensure("datasets","ncbi-datasets-cli"); ensure("mmseqs","mmseqs2")
print(subprocess.check_output(["datasets","--version"],text=True).strip())
print(subprocess.check_output(["mmseqs","version"],text=True).strip())

## 1. Source download and parsing

The source is NCBI Datasets human GRCh38.p14 (`GCF_000001405.40`) with `rna,gff3`; the package, CLI version, download time, and source URL are recorded in the manifest. NCBI data-use terms should be reviewed before redistribution.

In [ ]:
zip_path=DATA_DIR/"human_rna_gff3.zip"; extract=DATA_DIR/"extracted"
if not zip_path.exists():
    subprocess.run(["datasets","download","genome","accession",ACCESSION,"--include","rna,gff3","--filename",str(zip_path),"--no-progressbar"],check=True)
if not extract.exists():
    extract.mkdir(); zipfile.ZipFile(zip_path).extractall(extract)
files=list(extract.rglob("*")); fasta=next(p for p in files if p.name.endswith((".fna",".fna.gz"))); gff=next(p for p in files if p.name.endswith((".gff",".gff3",".gff.gz",".gff3.gz")))
def ot(p): return gzip.open(p,"rt") if str(p).endswith(".gz") else open(p)
def attrs(s):
    d={}
    for x in s.strip().split(";"):
        if "=" in x: k,v=x.split("=",1); d[k]=v
    return d
ann={}
with ot(gff) as fh:
    for line in fh:
        if line.startswith("#"): continue
        f=line.rstrip().split("	")
        if len(f)!=9: continue
        a=f[8]; aa=attrs(a); accs=set(re.findall(r"N[MR]_\d+(?:\.\d+)?",a)); gm=re.search(r"(?:GeneID:|gene_id=)(\d+)",a)
        for acc in accs: ann[acc.split(".")[0]]={"gene_id":gm.group(1) if gm else None,"gene_biotype":aa.get("gene_biotype") or aa.get("gene_type") or aa.get("biotype"),"gff_feature":f[2]}
def parse_header(h):
    m=re.search(r"(N[MR]_\d+)",h)
    if not m:return None
    acc=m.group(1); p=1 if acc.startswith("NM_") else 0
    gm=re.search(r"(?:GeneID:|gene_id[= ]?)(\d+)",h,re.I); gs=re.search(r"\[gene=([^\]]+)\]",h); hb=re.search(r"\[(?:gene_biotype|gene_type|biotype)=([^\]]+)\]",h,re.I)
    return {"accession":acc,"prefix_label":p,"gene_id_header":gm.group(1) if gm else None,"gene_symbol":gs.group(1) if gs else None,"header_biotype":hb.group(1) if hb else None,"header":h}
rows=[]; h=None; parts=[]
def flush():
    if h is None:return
    m=parse_header(h)
    if not m:return
    s="".join(parts).upper().replace("U","T"); m.update(ann.get(m["accession"],{})); m["sequence"]=s; m["length"]=len(s)
    m["gene_id"]=m.get("gene_id") or m.get("gene_id_header") or ("symbol:"+str(m.get("gene_symbol")))
    rows.append(m)
with ot(fasta) as fh:
    for line in fh:
        line=line.strip()
        if line.startswith(">"):
            flush(); h=line; parts=[]
        else: parts.append(line)
    flush()
df=pd.DataFrame(rows).drop_duplicates("accession").reset_index(drop=True)
df["label_name"]=df.prefix_label.map({1:"protein-coding",0:"non-coding"}); df["valid_bases"]=df.sequence.str.fullmatch(r"[ACGTN]+",na=False)
print("FASTA",fasta,"GFF3",gff,"records",len(df)); print(df.label_name.value_counts())

## 2. Data audit: counts, NR_ biotypes, and prefix/GFF mismatches

The audit is calculated on every parsed eligible record before any modeling sample is taken. `prefix_gff_mismatch` is intentionally visible: a mismatch is a review flag, not an automatic relabeling.

In [ ]:
audit={"NM_count":int((df.prefix_label==1).sum()),"NR_count":int((df.prefix_label==0).sum()),"duplicate_accession_rows":int(df.accession.duplicated().sum()),"missing_gff_biotype":int(df.gene_biotype.isna().sum()),"below_minimum_length":int((df.length<MIN_LENGTH_NT).sum()),"invalid_base_rows":int((~df.valid_bases).sum()),"missing_gene_id":int(df.gene_id.isna().sum())}
print(pd.Series(audit).to_string()); print("\nClass balance"); display(df.label_name.value_counts().rename_axis("label").to_frame("n").assign(fraction=lambda x:x.n/x.n.sum()))
print("\nNR_ gene_biotype values"); nr_biotypes=df.loc[df.prefix_label==0,"gene_biotype"].fillna("<missing GFF3 biotype>").value_counts().rename_axis("gene_biotype").to_frame("n"); nr_biotypes["fraction_within_NR"]=nr_biotypes.n/nr_biotypes.n.sum(); display(nr_biotypes)
def norm(x):
    if pd.isna(x):return None
    x=str(x).lower().replace("-","_").replace(" ","_")
    if x in {"protein_coding","protein_coding_gene"}:return "protein_coding"
    for key,name in [("lncrna","lncRNA"),("long_noncoding","lncRNA"),("mirna","miRNA"),("snrna","snRNA"),("snorna","snoRNA"),("trna","tRNA"),("rrna","rRNA")]:
        if key in x:return name
    return x
df["gff_biotype_normalized"]=df.gene_biotype.map(norm)
df["prefix_gff_mismatch"]=((df.prefix_label==1)&df.gff_biotype_normalized.notna()&(df.gff_biotype_normalized!="protein_coding"))|((df.prefix_label==0)&(df.gff_biotype_normalized=="protein_coding"))
print("Prefix/GFF mismatches:",int(df.prefix_gff_mismatch.sum())); display(df.loc[df.prefix_gff_mismatch,["accession","prefix_label","gene_id","gene_biotype","gff_feature","header"]].head(25))
model_pool=df[(df.length>=MIN_LENGTH_NT)&df.valid_bases&df.gene_id.notna()].copy(); print("Model pool after QC",len(model_pool))
display(model_pool.groupby("label_name").length.quantile([0,.01,.25,.5,.75,.99,1]).unstack(0))

## 3. GeneID-first, then MMseqs2 cluster-level split

All isoforms for a `GeneID` remain together. The longest transcript per gene is clustered with MMseqs2 at 80% nucleotide identity and 80% coverage; every transcript inherits that cluster. The split is at cluster level and asserts zero pairwise overlap for both `GeneID` and cluster ID.

In [ ]:
rng=np.random.default_rng(SEED); pieces=[]
for y in [0,1]:
    p=model_pool[model_pool.prefix_label==y]
    if MAX_MODEL_PER_CLASS and len(p)>MAX_MODEL_PER_CLASS:p=p.iloc[rng.choice(len(p),MAX_MODEL_PER_CLASS,replace=False)]
    pieces.append(p)
model_df=pd.concat(pieces).sample(frac=1,random_state=SEED).reset_index(drop=True)
reps=model_df.sort_values(["gene_id","length"],ascending=[True,False]).drop_duplicates("gene_id"); rep_fasta=DATA_DIR/"gene_representatives.fna"
with open(rep_fasta,"w") as out:
    for _,r in reps.iterrows():out.write(f">gene_{r.gene_id}\n{r.sequence}\n")
mm=DATA_DIR/"mmseqs"; mm.mkdir(exist_ok=True); pref=mm/"clusters"; tsv=Path(str(pref)+"_cluster.tsv")
if not tsv.exists():subprocess.run(["mmseqs","easy-cluster",str(rep_fasta),str(pref),str(mm/"tmp"),"--min-seq-id",str(MMSEQS_IDENTITY),"-c",str(MMSEQS_COVERAGE),"--cov-mode","0","--threads","2"],check=True)
cluster_map={}
for line in open(tsv):
    a,b=line.rstrip().split("	")[:2]; cluster_map[b.replace("gene_","",1)]=a.replace("gene_","",1)
model_df["cluster_id"]=model_df.gene_id.map(cluster_map); assert model_df.cluster_id.notna().all()
from sklearn.model_selection import GroupShuffleSplit
def split3(data,seed=SEED):
    for off in range(100):
        rs=seed+off; o=GroupShuffleSplit(1,test_size=TEST_SIZE,random_state=rs); a,b=next(o.split(data,groups=data.cluster_id)); trv=data.iloc[a]; te=data.iloc[b]; i=GroupShuffleSplit(1,test_size=VAL_SIZE,random_state=rs); c,d=next(i.split(trv,groups=trv.cluster_id)); tr=trv.iloc[c]; va=trv.iloc[d]
        if all(set(x.prefix_label)=={0,1} for x in [tr,va,te]):return tr.copy(),va.copy(),te.copy(),rs
    raise RuntimeError("No split with both classes")
train,val,test,split_seed=split3(model_df); sets={"train":train,"validation":val,"test":test}
for n,p in sets.items():print(n,len(p),"transcripts",p.gene_id.nunique(),"genes",p.cluster_id.nunique(),"clusters",p.label_name.value_counts().to_dict())
for a,b in [("train","validation"),("train","test"),("validation","test")]:print(a,b,"GeneID overlap",len(set(sets[a].gene_id)&set(sets[b].gene_id)),"cluster overlap",len(set(sets[a].cluster_id)&set(sets[b].cluster_id)))
assert all(len(set(sets[a].cluster_id)&set(sets[b].cluster_id))==0 and len(set(sets[a].gene_id)&set(sets[b].gene_id))==0 for a,b in [("train","validation"),("train","test"),("validation","test")]); print("PASS: zero cross-split overlap")

## 4. Models and validation-only tuning

The majority baseline is fit on training labels. The interpretable logistic regression uses length, GC, and normalized 3-mer frequencies. A character 3–5-mer TF-IDF logistic regression is tuned over a small `C` grid using validation AUPRC only; preprocessing is fit on training data.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix
kmers=[a+b+c for a in "ACGT" for b in "ACGT" for c in "ACGT"]
def feat(ss):
    out=[]
    for s in ss:
        d=max(1,len(s)-2); c=Counter(s[i:i+3] for i in range(len(s)-2)); out.append([len(s),(s.count("G")+s.count("C"))/max(1,len(s))]+[c[k]/d for k in kmers])
    return np.asarray(out,dtype="float32")
Xtr,Xv,Xte=[feat(p.sequence) for p in [train,val,test]]
majority=DummyClassifier(strategy="most_frequent").fit(np.zeros((len(train),1)),train.prefix_label)
interp=Pipeline([("scale",StandardScaler()),("logreg",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED))]).fit(Xtr,train.prefix_label)
candidates={}
for c in [.25,1.,4.]:
    m=Pipeline([("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=50000,sublinear_tf=True)),("logreg",LogisticRegression(max_iter=1000,C=c,class_weight="balanced",random_state=SEED))]).fit(train.sequence,train.prefix_label)
    candidates[c]=(average_precision_score(val.prefix_label,m.predict_proba(val.sequence)[:,1]),m)
tuned_c,(best_val_auprc,tuned)=max(candidates.items(),key=lambda x:x[1][0]); print({c:round(v[0],4) for c,v in candidates.items()},"selected C",tuned_c)
def row(name,y,p,s):return {"model":name,"precision":precision_score(y,p,zero_division=0),"recall":recall_score(y,p,zero_division=0),"f1":f1_score(y,p,zero_division=0),"auroc":roc_auc_score(y,s),"auprc":average_precision_score(y,s)}
vr=[]
for name,m,x in [("majority",majority,None),("length+GC+3mer logistic",interp,Xv),("TF-IDF logistic tuned",tuned,val.sequence)]:
    p=m.predict(np.zeros((len(val),1))) if x is None else m.predict(x); s=np.repeat(train.prefix_label.mean(),len(val)) if x is None else m.predict_proba(x)[:,1]; vr.append(row(name,val.prefix_label,p,s))
display(pd.DataFrame(vr).round(4))
trainval=pd.concat([train,val]); final=Pipeline([("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=50000,sublinear_tf=True)),("logreg",LogisticRegression(max_iter=1000,C=float(tuned_c),class_weight="balanced",random_state=SEED))]).fit(trainval.sequence,trainval.prefix_label)

## 5. Locked test metrics, bootstrap intervals, and seed sensitivity

The test set is touched only after model selection. The headline table includes precision, recall, F1, AUROC, AUPRC, and 95% bootstrap intervals; the confusion matrix is printed. Seed sensitivity is evaluated on validation only so the test remains a one-time locked evaluation.

In [ ]:
def ci(y,s,n=500):
    rng=np.random.default_rng(SEED); z=[]
    for _ in range(n):
        i=rng.integers(0,len(y),len(y)); yy=np.asarray(y)[i]; ss=np.asarray(s)[i]
        if len(np.unique(yy))<2:continue
        pp=(ss>=.5).astype(int); z.append([precision_score(yy,pp,zero_division=0),recall_score(yy,pp,zero_division=0),f1_score(yy,pp,zero_division=0),roc_auc_score(yy,ss),average_precision_score(yy,ss)])
    q=np.quantile(z,[.025,.975],axis=0); return [f"[{q[0,i]:.3f}, {q[1,i]:.3f}]" for i in range(5)]
test_rows=[]; predictions={}
for name,m,x in [("majority",majority,None),("length+GC+3mer logistic",interp,Xte),("TF-IDF logistic locked",final,test.sequence)]:
    p=m.predict(np.zeros((len(test),1))) if x is None else m.predict(x); s=np.repeat(trainval.prefix_label.mean(),len(test)) if x is None else m.predict_proba(x)[:,1]; predictions[name]=(p,s); r=row(name,test.prefix_label,p,s); q=ci(test.prefix_label,s)
    for k,v in zip(["precision","recall","f1","auroc","auprc"],q):r[k+"_ci95"]=v
    test_rows.append(r)
test_results=pd.DataFrame(test_rows); display(test_results.round(4)); pred,score=predictions["TF-IDF logistic locked"]; print("Confusion matrix [rows=true, cols=predicted]\n",confusion_matrix(test.prefix_label,pred))
seed_rows=[]
for seed in [42,43,44]:
    m=Pipeline([("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=50000,sublinear_tf=True)),("logreg",LogisticRegression(max_iter=1000,C=float(tuned_c),class_weight="balanced",random_state=seed))]).fit(train.sequence,train.prefix_label); s=m.predict_proba(val.sequence)[:,1]; seed_rows.append({"seed":seed,"validation_auroc":roc_auc_score(val.prefix_label,s),"validation_auprc":average_precision_score(val.prefix_label,s)})
display(pd.DataFrame(seed_rows).round(4))

## 6. Error analysis and limitations

The following tables break errors down by transcript length, GC content, and GFF3 biotype. Inspect short-RNA rows and prefix/GFF mismatches together: a model can exploit real distributional differences without learning biological coding function. Results are limited to this RefSeq sampling frame and the operational labels; a bioinformatics reviewer should validate the release, annotation semantics, cluster threshold, and external generalization.

In [ ]:
err=test[["accession","gene_id","gene_biotype","label_name","length","sequence","header"]].copy(); err["gc_fraction"]=err.sequence.map(lambda s:(s.count("G")+s.count("C"))/max(1,len(s))); err["predicted_label"]=np.where(pred==1,"protein-coding","non-coding"); err["coding_probability"]=score; err["correct"]=err.label_name==err.predicted_label; err["error_type"]=np.select([(~err.correct)&(err.label_name=="non-coding"),(~err.correct)&(err.label_name=="protein-coding")],["false positive","false negative"],default="correct")
print("Overall error rate",round(1-err.correct.mean(),4)); display(err.groupby("label_name").agg(n=("correct","size"),error_rate=("correct",lambda x:1-x.mean()),median_length=("length","median"),median_gc=("gc_fraction","median")).round(4)); print("By GFF3 biotype (n>=5)"); display(err.groupby(err.gene_biotype.fillna("<missing>")).agg(n=("correct","size"),error_rate=("correct",lambda x:1-x.mean()),median_length=("length","median"),median_gc=("gc_fraction","median")).query("n>=5").sort_values("error_rate",ascending=False).round(4)); err["length_bin"]=pd.qcut(err.length,4,duplicates="drop"); print("By length quartile"); display(err.groupby("length_bin",observed=True).agg(n=("correct","size"),error_rate=("correct",lambda x:1-x.mean()),median_gc=("gc_fraction","median")).round(4)); err["uncertainty"]=(err.coding_probability-.5).abs(); print("Most uncertain errors"); display(err[~err.correct].sort_values("uncertainty")[["accession","gene_id","gene_biotype","label_name","predicted_label","coding_probability","length","gc_fraction","header"]].head(25))
manifest={"accession":ACCESSION,"expected_assembly":EXPECTED_ASSEMBLY,"expected_annotation_release":EXPECTED_ANNOTATION_RELEASE,"source_url":"https://www.ncbi.nlm.nih.gov/datasets/genome/"+ACCESSION+"/","download_timestamp_utc":datetime.now(timezone.utc).isoformat(),"datasets_version":subprocess.check_output(["datasets","--version"],text=True).strip(),"mmseqs_version":subprocess.check_output(["mmseqs","version"],text=True).strip(),"min_length_nt":MIN_LENGTH_NT,"max_model_per_class":MAX_MODEL_PER_CLASS,"mmseqs_min_seq_id":MMSEQS_IDENTITY,"mmseqs_min_coverage":MMSEQS_COVERAGE,"audit":audit,"nr_biotypes":nr_biotypes.reset_index().to_dict("records"),"prefix_gff_mismatches":int(df.prefix_gff_mismatch.sum()),"split_seed":int(split_seed),"train_n":len(train),"validation_n":len(val),"test_n":len(test),"cross_split_gene_overlap":0,"cross_split_cluster_overlap":0}
Path("transcript_coding_status_manifest.json").write_text(json.dumps(manifest,indent=2,default=str)); print(json.dumps(manifest,indent=2,default=str))

## 7. MMseqs2 identity-threshold sensitivity: 70% to 95%

The primary split uses 80% identity. This sensitivity analysis repeats the cluster construction and grouped split at **70%, 75%, 80%, 85%, 90%, and 95% nucleotide identity**, keeping coverage at 80%. For every split threshold, the notebook reports the number of shared clusters across train/test under each audit threshold. This makes both the guaranteed zero-overlap condition and residual leakage under looser similarity definitions visible.

For generalization, the same locked TF-IDF model family and the `C` selected in the primary validation split are refit separately for each threshold split. These are threshold-sensitivity estimates, not additional tuning on the test data.

In [ ]:
THRESHOLDS=[.70,.75,.80,.85,.90,.95]

def cluster_genes_at_threshold(threshold):
    tag=f"{int(round(threshold*100)):02d}"; pref=mm/f"clusters_{tag}"; out_tsv=Path(str(pref)+"_cluster.tsv")
    if not out_tsv.exists():
        subprocess.run(["mmseqs","easy-cluster",str(rep_fasta),str(pref),str(mm/f"tmp_{tag}"),"--min-seq-id",str(threshold),"-c",str(MMSEQS_COVERAGE),"--cov-mode","0","--threads","2"],check=True)
    mapping={}
    for line in open(out_tsv):
        a,b=line.rstrip().split("\t")[:2]; mapping[b.replace("gene_","",1)]=a.replace("gene_","",1)
    return mapping

cluster_maps={t:cluster_genes_at_threshold(t) for t in THRESHOLDS}
for t,mapping in cluster_maps.items():
    model_df[f"cluster_{int(t*100)}"]=model_df.gene_id.map(mapping)
    assert model_df[f"cluster_{int(t*100)}"].notna().all()
print("Clusters by identity threshold:", {int(t*100):model_df[f"cluster_{int(t*100)}"].nunique() for t in THRESHOLDS})

def split_by_group(data, group_col, seed=SEED):
    for off in range(100):
        rs=seed+off; outer=GroupShuffleSplit(1,test_size=TEST_SIZE,random_state=rs); a,b=next(outer.split(data,groups=data[group_col])); trv=data.iloc[a]; te=data.iloc[b]
        inner=GroupShuffleSplit(1,test_size=VAL_SIZE,random_state=rs); c,d=next(inner.split(trv,groups=trv[group_col])); tr=trv.iloc[c]; va=trv.iloc[d]
        if all(set(x.prefix_label)=={0,1} for x in [tr,va,te]): return tr.copy(),va.copy(),te.copy(),rs
    raise RuntimeError(f"No valid split for {group_col}")

threshold_splits={}
for t in THRESHOLDS:
    col=f"cluster_{int(t*100)}"; threshold_splits[t]=split_by_group(model_df,col,split_seed)

# Leakage matrix: for each split threshold, count shared clusters at each audit threshold.
leakage_rows=[]
for split_t,(tr,va,te,rs) in threshold_splits.items():
    for audit_t in THRESHOLDS:
        c=f"cluster_{int(audit_t*100)}"
        leakage_rows.append({"split_identity":split_t,"audit_identity":audit_t,"shared_train_test_clusters":len(set(tr[c])&set(te[c])),"shared_train_validation_clusters":len(set(tr[c])&set(va[c])),"shared_validation_test_clusters":len(set(va[c])&set(te[c]))})
leakage_matrix=pd.DataFrame(leakage_rows)
print("Cross-split cluster overlap matrix (train/test):")
display(leakage_matrix.pivot(index="split_identity",columns="audit_identity",values="shared_train_test_clusters").rename_axis(index="split %",columns="audit %"))
assert (leakage_matrix.query("split_identity == audit_identity").shared_train_test_clusters==0).all()

# Generalization comparison with a fixed model-selection choice from the primary workflow.
sweep_rows=[]
for t,(tr,va,te,rs) in threshold_splits.items():
    fit=pd.concat([tr,va],ignore_index=True)
    m=Pipeline([("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=50000,sublinear_tf=True)),("logreg",LogisticRegression(max_iter=1000,C=float(tuned_c),class_weight="balanced",random_state=SEED))]).fit(fit.sequence,fit.prefix_label)
    s=m.predict_proba(te.sequence)[:,1]; y=te.prefix_label; pred_t=(s>=.5).astype(int); audit_col=f"cluster_{int(t*100)}"
    own_overlap=len(set(tr[audit_col])&set(te[audit_col]))
    loose70=len(set(tr["cluster_70"])&set(te["cluster_70"]))
    sweep_rows.append({"split_identity":t,"clusters":model_df[audit_col].nunique(),"train_n":len(tr),"validation_n":len(va),"test_n":len(te),"own_threshold_train_test_overlap":own_overlap,"70pct_train_test_overlap":loose70,"test_precision":precision_score(y,pred_t,zero_division=0),"test_recall":recall_score(y,pred_t,zero_division=0),"test_f1":f1_score(y,pred_t,zero_division=0),"test_auroc":roc_auc_score(y,s),"test_auprc":average_precision_score(y,s)})
threshold_results=pd.DataFrame(sweep_rows).sort_values("split_identity")
display(threshold_results.round(4))
assert (threshold_results.own_threshold_train_test_overlap==0).all()

# Add the sweep to the reproducibility manifest.
manifest["mmseqs_thresholds_tested"]=THRESHOLDS
manifest["threshold_leakage_matrix"]=leakage_matrix.to_dict("records")
manifest["threshold_generalization_results"]=threshold_results.to_dict("records")
Path("transcript_coding_status_manifest.json").write_text(json.dumps(manifest,indent=2,default=str))